# STKI RAG Demo\nDemo Retrieval Augmented Generation (RAG) untuk mata kuliah STKI.\nProject ini membandingkan 3 pendekatan retrieval semantik:\n1. Prompt-based (in-context LLM reasoning)\n2. Vector search (embedding + cosine similarity)\n3. KL-Divergence (softmax probability distribution)\n\nMenggunakan model Google Gemini untuk embedding dan generasi jawaban.

## Cell 1: Setup\nImport libraries and initialize Gemini client

In [ ]:
import os\nimport time\nimport hashlib\nfrom typing import Iterable\n\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom sklearn.decomposition import PCA\nfrom dotenv import load_dotenv\nfrom google import genai\n\n# Load .env and get API key\nload_dotenv()\nGEMINI_API_KEY = os.getenv("GEMINI_API_KEY")\n\nif not GEMINI_API_KEY:\n    raise EnvironmentError(\n        "GEMINI_API_KEY tidak ditemukan. "\n        "Isi file .env dengan GEMINI_API_KEY=key-mu."\n    )\n\nclient = genai.Client(api_key=GEMINI_API_KEY)

## Cell 2: Helper Functions\nDua fungsi utama yang dipakai berulang kali di notebook ini:\n- `get_embedding` — ubah teks menjadi vektor (dengan retry + fallback lokal)\n- `cosine_sim` — hitung cosine similarity antar dua vektor

In [ ]:
def _local_fallback_embedding(text: str, dim: int = 256) -> list[float]:\n    """Deterministic fallback when Gemini API is unreachable.\n    \n    Hashes each token into a sparse vector, then normalizes.\n    Not semantically accurate, but keeps the demo running offline.\n    """\n    vec = np.zeros(dim, dtype=np.float32)\n    tokens: Iterable[str] = text.lower().split()\n    for tok in tokens:\n        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)\n        vec[h % dim] += 1.0\n    norm = np.linalg.norm(vec)\n    if norm == 0:\n        return vec.tolist()\n    return (vec / norm).tolist()\n\n\ndef get_embedding(text: str, retries: int = 4, base_delay: float = 1.0) -> list[float]:\n    """Call Gemini embedding model with exponential backoff.\n    \n    Args:\n        text: Input text.\n        retries: Max attempts before falling back to local embedding.\n        base_delay: Starting delay in seconds for backoff.\n    \n    Returns:\n        Embedding vector as list of floats.\n    """\n    last_exc = None\n    for attempt in range(retries):\n        try:\n            resp = client.models.embed_content(\n                model="gemini-embedding-001",\n                contents=text,\n            )\n            return resp.embeddings[0].values\n        except Exception as exc:  # catches network/IPC/API errors\n            last_exc = exc\n            if attempt < retries - 1:\n                time.sleep(base_delay * (2 ** attempt))\n            continue\n\n    print(\n        f"Warning: embedding gagal setelah {retries} percobaan ({last_exc}). "\n        f"Pakai fallback lokal."\n    )\n    return _local_fallback_embedding(text)\n\n\ndef cosine_sim(a, b) -> float:\n    """Cosine similarity between two vectors.\n    \n    Returns 0.0 if either vector is zero-length to avoid division by zero.\n    """\n    a, b = np.array(a), np.array(b)\n    denom = np.linalg.norm(a) * np.linalg.norm(b)\n    if denom == 0:\n        return 0.0\n    return float(np.dot(a, b) / denom)

## Cell 3: Load Query and Documents\nTentukan query (pertanyaan) dan muat dokumen dari folder `dokumen/`.

In [ ]:
query = "What time do I leave for school?"\n\ndef load_documents() -> dict[str, str]:\n    """Read all markdown files from dokumen/ directory."""\n    docs = {}\n    for fname in ["D1.md", "D2.md", "D3.md"]:\n        with open(os.path.join("dokumen", fname), encoding="utf-8") as f:\n            docs[fname[:-3]] = f.read().strip()\n    return docs\n\nDOCS = load_documents()\n\nprint("Query:", query)\nprint("Documents loaded:", list(DOCS.keys()))\nfor name, text in DOCS.items():\n    print(f"--- {name} ---")\n    print(text[:120] + ("..." if len(text) > 120 else ""))\n    print()

## Cell 4: Vector Search\nLangkah-langkah:\n1. Embed semua dokumen menjadi vektor.\n2. Embed query menjadi vektor.\n3. Hitung cosine similarity antara query dan setiap dokumen.\n4. Pilih dokumen dengan skor tertinggi.

In [ ]:
doc_vecs = {}\nfor k, v in DOCS.items():\n    try:\n        doc_vecs[k] = get_embedding(v)\n        print(f"Embedded {k}: {len(doc_vecs[k])} dimensions")\n    except Exception as exc:\n        print(f"Skip dokumen {k} karena error: {exc}")\n\nif not doc_vecs:\n    raise RuntimeError("Tidak ada embedding dokumen yang berhasil dibuat.")\n\nq_vec = get_embedding(query)\nprint(f"Query embedded: {len(q_vec)} dimensions")\nprint()\n\nscores = {k: cosine_sim(q_vec, v) for k, v in doc_vecs.items()}\nbest = max(scores, key=scores.get)\n\nprint("Similarity scores:")\nfor k, v in sorted(scores.items(), key=lambda x: x[1], reverse=True):\n    print(f"  {k}: {v:.4f}")\nprint()\nprint(f"Best match: {best}")

## Cell 5: RAG Answer Generation\nDengan dokumen terbaik dari cosine similarity, kirim ke Gemini untuk generate\njawaban terstruktur: ANSWER / SNIPPET / REASON.

In [ ]:
METRIC = "cosine similarity"  # bisa diganti ke "KL-Divergence"\n\ndoc_text = DOCS[best]\nscores_text = "\n".join([f"- {k}: {v:.4f}" for k, v in scores.items()])\n\nprompt = (\n    "You are a semantic Question Answering system.\n"\n    f'A user asked: "{query}"\n\n'\n    f"Based on {METRIC}, the best matching document is {best}.\n"\n    f"Scores:\n{scores_text}\n\n"\n    f"Content of {best}:\n{doc_text}\n\n"\n    "Answer the query using ONLY the content above.\n"\n    "Format your answer as:\n"\n    f"ANSWER: [direct answer based on {best}]\n"\n    "SNIPPET: [1 relevant sentence from the document]\n"\n    "REASON: [why this document is the best match in 1 sentence]"\n)\n\nprint("Generating answer from", best, "...")\nstart_answer = time.time()\nresp = client.models.generate_content(\n    model="gemini-3.1-flash-lite",\n    contents=prompt,\n)\nelapsed_answer = time.time() - start_answer\n\n# Parse structured output\nlines = resp.text.strip().split("\n")\nrag_result = {"answer": "", "snippet": "", "reason": ""}\ncurrent = ""\nfor line in lines:\n    if line.startswith("ANSWER:"):\n        current = "answer"\n        rag_result["answer"] = line.replace("ANSWER:", "").strip()\n    elif line.startswith("SNIPPET:"):\n        current = "snippet"\n        rag_result["snippet"] = line.replace("SNIPPET:", "").strip()\n    elif line.startswith("REASON:"):\n        current = "reason"\n        rag_result["reason"] = line.replace("REASON:", "").strip()\n    elif current == "answer" and line.strip():\n        rag_result["answer"] += " " + line.strip()\n    elif current == "reason" and line.strip():\n        rag_result["reason"] += " " + line.strip()\n\nprint("\nANSWER:", rag_result["answer"])\nprint("SNIPPET:", rag_result["snippet"])\nprint("REASON:", rag_result["reason"])\nprint(f"[Generated in {elapsed_answer:.2f}s]")

## Cell 6: 2D Visualization\nGunakan PCA untuk reduksi vektor 768 dimensi jadi 2D, lalu plot dokumen + query.

In [ ]:
vectors = np.array([doc_vecs["D1"], doc_vecs["D2"], doc_vecs["D3"], q_vec])\nlabels = ["D1", "D2", "D3", "Query"]\n\npca = PCA(n_components=2)\ncoords = pca.fit_transform(vectors)\n\nplt.figure(figsize=(6, 6))\nfor i, label in enumerate(labels):\n    plt.scatter(coords[i, 0], coords[i, 1])\n    plt.text(coords[i, 0] + 0.01, coords[i, 1] + 0.01, label)\n\nplt.title("Embedding Space (PCA 2D)")\nplt.xlabel("PC1")\nplt.ylabel("PC2")\nplt.grid(True)\nplt.show()

## Cell 7: KL-Divergence Route\nAlternatif measurement selain cosine similarity:\n- Ubah vektor menjadi probability distribution dengan softmax.\n- Hitung KL-Divergence antara query dan setiap dokumen.\n- Semakin kecil KL, semakin similar.

In [ ]:
def softmax(x) -> np.ndarray:\n    """Convert vector to probability distribution."""\n    x = np.array(x, dtype=np.float64)\n    x = x - np.max(x)  # numerical stability\n    e_x = np.exp(x)\n    return e_x / e_x.sum()\n\n\ndef kl_divergence(p, q, eps: float = 1e-10) -> float:\n    """Compute D_KL(P || Q). Lower means more similar."""\n    p, q = np.array(p) + eps, np.array(q) + eps\n    return float(np.sum(p * np.log(p / q)))\n\n\nq_prob = softmax(q_vec)\nkl_scores = {k: kl_divergence(q_prob, softmax(v)) for k, v in doc_vecs.items()}\nkl_best = min(kl_scores, key=kl_scores.get)\n\nprint("KL-Divergence scores (lower = more similar):")\nfor k, v in sorted(kl_scores.items(), key=lambda x: x[1]):\n    print(f"  {k}: {v:.6f}")\nprint()\nprint("Best match:", kl_best)

## Cell 8: Summary Comparison\nBandingkan peringkat dari cosine similarity vs KL-Divergence.

In [ ]:
print("=" * 40)\nprint("SUMMARY")\nprint("=" * 40)\nprint(f"Query           : {query}")\nprint()\nprint(f"Cosine best     : {best}  (score: {scores[best]:.4f})")\nprint(f"KL best         : {kl_best}  (score: {kl_scores[kl_best]:.6f})")\nprint()\nprint("Cosine ranking:")\nfor rank, (k, v) in enumerate(sorted(scores.items(), key=lambda x: x[1], reverse=True), 1):\n    print(f"  {rank}. {k}: {v:.4f}")\nprint()\nprint("KL-Divergence ranking:")\nfor rank, (k, v) in enumerate(sorted(kl_scores.items(), key=lambda x: x[1]), 1):\n    print(f"  {rank}. {k}: {v:.6f}")